# AC-MOT — Run All: Optuna SCI-only tuning

هذا الـ (Notebook) جاهز لـ **Run All**.

المسارات المثبتة:
- Validation: `/content/drive/MyDrive/AC-MOT-data/VisDrone2019-MOT-val`
- Test-dev: `/content/drive/MyDrive/VisDrone2019-MOT-test-dev`

التجربة:
1. تثبيت (YOLOv8n) بدون تدريب.
2. تثبيت (ByteTrack).
3. تشغيل (Old A3) على مجموعة التحقق.
4. تحسين أوزان مؤشر تعقيد المشهد (SCI) فقط باستخدام (Optuna).
5. قبول الحلول التي تحقق سرعة ≥ 25 إطار/ثانية وألا تزيد تبديلات الهوية (IDS) عن النسخة القديمة.
6. اختيار أعلى (MOTA)، ثم أقل (IDS)، ثم أعلى (FPS).
7. تجميد الأوزان.
8. تشغيل الاختبار النهائي مرة واحدة.
9. إذا شغلت **Run All** لاحقًا ووجدت نتيجة اختبار نهائي محفوظة، لن يعيد اختبار (Test-dev).

لا يوجد تدريب للكاشف في هذا الـ Notebook.


In [ ]:
# [1] Configuration
from pathlib import Path

REPO_URL = "https://github.com/AhmedCode110/AC-MOT.git"
REPO_DIR = Path("/content/AC-MOT")

DATA_ROOT = Path("/content/drive/MyDrive/AC-MOT-data")
RESULT_ROOT = Path("/content/drive/MyDrive/AC-MOT-results/optuna_sci_only")

# Correct permanent Google Drive locations
VAL_DIR = Path("/content/drive/MyDrive/AC-MOT-data/VisDrone2019-MOT-val")
TEST_DIR = Path("/content/drive/MyDrive/VisDrone2019-MOT-test-dev")

N_TRIALS = 30
SEED = 42
MIN_FPS = 25.0
PROGRESS_EVERY = 250

# Selected Optuna solution must not have more IDS than old A3 on validation
ENFORCE_IDS_NOT_WORSE_THAN_OLD_A3 = True

print("Validation:", VAL_DIR)
print("Test:", TEST_DIR)
print("Optuna trials:", N_TRIALS)


In [ ]:
# [2] Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print("[OK] Drive mounted")


In [ ]:
# [3] Fresh clone + dependencies + pinned TrackEval + fixed YOLOv8n weights
import os, sys, shutil, subprocess
from pathlib import Path

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    check=True
)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.3.200",
    "numpy==2.2.6",
    "scipy==1.15.3",
    "lap",
    "opencv-python-headless",
    "optuna>=4,<5",
    "gdown",
    "pandas"
], check=True)

TRACKEVAL_DIR = Path("/content/TrackEval")
if TRACKEVAL_DIR.exists():
    shutil.rmtree(TRACKEVAL_DIR)

subprocess.run(
    ["git", "clone", "-q", "https://github.com/JonathonLuiten/TrackEval.git", str(TRACKEVAL_DIR)],
    check=True
)
subprocess.run(
    ["git", "-C", str(TRACKEVAL_DIR), "checkout", "-q",
     "12c8791b303e0a0b50f753af204249e622d0281a"],
    check=True
)

WEIGHTS_DIR = Path("/content/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS = WEIGHTS_DIR / "yolov8n.pt"

if not WEIGHTS.exists():
    from ultralytics import YOLO
    old_cwd = os.getcwd()
    os.chdir(WEIGHTS_DIR)
    try:
        YOLO("yolov8n.pt")
    finally:
        os.chdir(old_cwd)

if not WEIGHTS.exists():
    raise RuntimeError("Could not obtain fixed yolov8n.pt")

print("[OK] Repo commit:", commit)
print("[OK] Fixed detector weights:", WEIGHTS)
print("[OK] NO detector training will be performed")


In [ ]:
# [4] Verify permanent datasets on Google Drive
#
# Validation:
# - If already present, use it directly.
# - If missing, download the VisDrone2019-MOT-val mirror once to Google Drive,
#   extract it, verify it, then delete the ZIP.
#
# Test-dev:
# - Expected at the user's existing permanent location:
#   /content/drive/MyDrive/VisDrone2019-MOT-test-dev
# - It is NOT downloaded or moved.

import subprocess, zipfile, shutil
from pathlib import Path

VAL_MIRROR_URL = (
    "https://huggingface.co/datasets/vanthanh/VisDrone2019-MOT/"
    "resolve/main/VisDrone2019-MOT-val.zip"
)

def valid_visdrone_root(path: Path):
    return (
        path.is_dir()
        and (path / "sequences").is_dir()
        and (path / "annotations").is_dir()
    )

def count_dataset(path: Path):
    seqs = sorted(p for p in (path / "sequences").iterdir() if p.is_dir())
    frames = sum(len(list(p.glob("*.jpg"))) for p in seqs)
    anns = len([
        p for p in (path / "annotations").glob("*.txt")
        if not p.stem.endswith("_clean")
    ])
    return len(seqs), frames, anns

# Validation: persistent on Drive.
if valid_visdrone_root(VAL_DIR):
    print("[EXISTS] Validation:", VAL_DIR)
else:
    VAL_DIR.parent.mkdir(parents=True, exist_ok=True)
    val_zip = VAL_DIR.parent / "VisDrone2019-MOT-val.zip"

    print("[DOWNLOAD] Validation -> Google Drive")
    subprocess.run([
        "wget", "-c",
        "-O", str(val_zip),
        VAL_MIRROR_URL
    ], check=True)

    print("[EXTRACT] Validation")
    with zipfile.ZipFile(val_zip, "r") as z:
        z.extractall(VAL_DIR.parent)

    if not valid_visdrone_root(VAL_DIR):
        raise RuntimeError(
            "Validation download completed, but expected structure was not found at "
            f"{VAL_DIR}"
        )

    val_zip.unlink(missing_ok=True)
    print("[OK] Validation saved permanently on Google Drive")

# Test-dev: use the existing permanent copy.
if not valid_visdrone_root(TEST_DIR):
    raise RuntimeError(
        "Test-dev was not found at the expected existing path:\n"
        f"{TEST_DIR}\n"
        "Expected subfolders: sequences/ and annotations/"
    )

val_info = count_dataset(VAL_DIR)
test_info = count_dataset(TEST_DIR)

print("[VAL]  sequences, frames, annotations =", val_info)
print("[TEST] sequences, frames, annotations =", test_info)

if val_info[0] != 7:
    raise RuntimeError(f"Expected 7 validation sequences, found {val_info[0]}")
if test_info[0] != 17:
    raise RuntimeError(f"Expected 17 test-dev sequences, found {test_info[0]}")

print("[OK] Dataset paths are ready for Run All")


In [ ]:
# [5] GPU + repository preflight
import sys, json, torch, numpy as np, pandas as pd, csv, gzip, gc

sys.path.insert(0, str(REPO_DIR))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required. Select T4 GPU in Colab.")

GPU_NAME = torch.cuda.get_device_name(0)
print("[GPU]", GPU_NAME)

if "T4" not in GPU_NAME:
    print("[WARNING] FPS will not be directly comparable with previous T4 runs.")

from experiment import dataset_manifest
from core_v17 import PresentationController as OldPresentationController
from core_v17 import PresentationSpec
from core import boxes

val_sequences = sorted(p.name for p in (VAL_DIR / "sequences").iterdir() if p.is_dir())
test_sequences = sorted(p.name for p in (TEST_DIR / "sequences").iterdir() if p.is_dir())

VAL_MANIFEST = dataset_manifest(VAL_DIR, val_sequences)
TEST_MANIFEST = dataset_manifest(TEST_DIR, test_sequences)

print("[OK] Validation sequences:", len(VAL_MANIFEST))
print("[OK] Test sequences:", len(TEST_MANIFEST))


In [ ]:
# [6] Tunable SCI controller
#
# Only these five coefficients change.
# Detector, tracker, scene cues, smoothing, analysis stride, and SmartCalibrator
# remain fixed to isolate the effect of the SCI equation.

from collections import deque
import numpy as np

class TunableSCIController:
    def __init__(self, spec: PresentationSpec, weights: dict):
        self.spec = spec.validate()
        self.history = deque(maxlen=spec.smoothing_window)
        self.sci = 0.0
        self.tiny = 0.0
        self.scene = "clear"

        keys = ["crowd", "tiny", "edge", "night", "blur"]
        vals = np.array([float(weights[k]) for k in keys], dtype=float)

        if np.any(vals < 0) or vals.sum() <= 0:
            raise ValueError("SCI weights must be non-negative and non-zero")

        vals = vals / vals.sum()
        self.weights = dict(zip(keys, vals.tolist()))

        self.last_params = dict(
            conf=0.25, nms=0.45, size=640, sci=0.0, scene="clear"
        )

    def choose(self, frame, visual, previous):
        s = self.spec
        analyze = frame == 1 or (frame - 1) % s.analysis_stride == 0

        if analyze:
            previous = boxes(previous)
            n = len(previous)

            self.tiny = float(np.mean(
                (previous[:, 2] - previous[:, 0]) *
                (previous[:, 3] - previous[:, 1]) < 32 * 32
            )) if n else 0.0

            crowd = min(n / 30.0, 1.0)
            edge = float(visual["edges"])
            brightness = float(visual["brightness"])
            blur_value = float(visual["blur"])

            edge_norm = min(edge / 0.14, 1.0)
            night = 1.0 if brightness < 80 else 0.0
            blur_flag = 1.0 if blur_value < 180 else 0.0

            w = self.weights
            raw = (
                w["crowd"] * crowd
                + w["tiny"] * self.tiny
                + w["edge"] * edge_norm
                + w["night"] * night
                + w["blur"] * blur_flag
            )

            self.history.append(float(np.clip(raw, 0.0, 1.0)))
            self.sci = float(np.mean(self.history))

            if brightness < 80:
                self.scene = "night"
            elif blur_value < 180:
                self.scene = "blur"
            elif self.tiny > 0.50:
                self.scene = "tiny"
            elif crowd > 0.65 or edge > 0.13:
                self.scene = "crowded"
            else:
                self.scene = "clear"

        # Current SmartCalibrator mapping stays unchanged.
        conf, nms, size = 0.25, 0.45, 640

        if s.adaptive_threshold:
            conf = 0.245 - 0.050 * self.sci
            nms = 0.490 - 0.050 * self.sci

            if self.scene in {"crowded", "tiny", "night"}:
                conf -= 0.012
            if self.scene == "blur":
                nms -= 0.012

            conf = float(np.clip(conf, 0.19, 0.28))
            nms = float(np.clip(nms, 0.40, 0.52))

        if s.adaptive_resolution:
            if self.sci > 0.60 or self.tiny > 0.50:
                size = 832
            elif self.sci > 0.35 or self.scene in {"crowded", "tiny"}:
                size = 736

        self.last_params = dict(
            conf=float(conf),
            nms=float(nms),
            size=int(size),
            sci=float(self.sci),
            scene=self.scene,
        )
        return dict(self.last_params)

def normalized_weights_from_trial(trial):
    raw = {
        "crowd": trial.suggest_float("raw_crowd", 0.02, 1.0),
        "tiny": trial.suggest_float("raw_tiny", 0.02, 1.0),
        "edge": trial.suggest_float("raw_edge", 0.02, 1.0),
        "night": trial.suggest_float("raw_night", 0.02, 1.0),
        "blur": trial.suggest_float("raw_blur", 0.02, 1.0),
    }
    total = sum(raw.values())
    return {k: v / total for k, v in raw.items()}

def normalized_weights_from_params(params):
    raw = {
        "crowd": params["raw_crowd"],
        "tiny": params["raw_tiny"],
        "edge": params["raw_edge"],
        "night": params["raw_night"],
        "blur": params["raw_blur"],
    }
    total = sum(raw.values())
    return {k: v / total for k, v in raw.items()}


In [ ]:
# [7] Common runner using the existing AC-MOT v17 timing and TrackEval path

import scripts.paper_eval_v17 as pe
import subprocess, shutil, json
from pathlib import Path

GT_FILTER = dict(categories=[1, 4, 5, 6, 9], score=1, occlusion_lt=2, truncation_lt=2)

A3_SYSTEM_TEMPLATE = {
    "name": "SYSTEM",
    "tracker_profile": "tuned",
    "adaptive_threshold": True,
    "adaptive_resolution": True,
    "smoothing_window": 7,
    "analysis_stride": 10,
}

A0_SYSTEM_TEMPLATE = {
    "name": "SYSTEM",
    "tracker_profile": "default",
    "adaptive_threshold": False,
    "adaptive_resolution": False,
    "smoothing_window": 7,
    "analysis_stride": 10,
}

def write_eval_metadata(run_root: Path, manifest, systems):
    (run_root / "configuration.json").write_text(json.dumps({
        "version": "optuna_sci_only",
        "purpose": "SCI-only validation tuning with fixed detector and tracker",
        "systems": systems,
        "ground_truth_filter": GT_FILTER,
    }, indent=2))
    (run_root / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))

def parse_metrics(summary_csv: Path, system_name: str):
    rows = list(csv.DictReader(summary_csv.open()))
    row = next(r for r in rows if r["system"] == system_name)

    return {
        "HOTA": float(row["HOTA"]),
        "DetA": float(row["DetA"]),
        "AssA": float(row["AssA"]),
        "MOTA": float(row["MOTA"]),
        "IDF1": float(row["IDF1"]),
        "IDS": int(float(row["IDS"])),
        "FN": int(float(row["FN"])),
        "FP": int(float(row["FP"])),
    }

def run_single_system(
    *,
    dataset: Path,
    manifest,
    system: dict,
    run_root: Path,
    controller_factory,
    keep_tracks: bool = False,
):
    if run_root.exists():
        shutil.rmtree(run_root)

    run_root.mkdir(parents=True, exist_ok=False)
    write_eval_metadata(run_root, manifest, [system])

    original_controller = pe.PresentationController

    try:
        pe.PresentationController = controller_factory

        timing = pe.run_system(
            system=system,
            dataset=dataset,
            manifest=manifest,
            weights=WEIGHTS,
            engine=Path("/content/weights/unused.engine"),
            target_fps=MIN_FPS,
            progress_every=PROGRESS_EVERY,
            backend_pref="pytorch",
            decode_chunk_size=16,
            output=run_root,
            index=1,
            total=1,
        )

    finally:
        pe.PresentationController = original_controller

    eval_out = run_root / "trackeval"

    subprocess.run([
        sys.executable,
        str(REPO_DIR / "evaluate.py"),
        str(run_root),
        "--dataset", str(dataset),
        "--trackeval", str(TRACKEVAL_DIR),
        "--output", str(eval_out),
    ], cwd=REPO_DIR, check=True)

    metrics = parse_metrics(eval_out / "summary.csv", system["name"])

    metrics["FPS"] = float(timing["processing_fps"])
    metrics["mean_imgsz"] = float(timing["mean_imgsz"])
    metrics["mean_conf"] = float(timing["mean_conf"])
    metrics["mean_nms_iou"] = float(timing["mean_nms_iou"])

    if not keep_tracks:
        (run_root / "compact_result.json").write_text(json.dumps(metrics, indent=2))

        for p in list(run_root.iterdir()):
            if p.name not in {"compact_result.json", "trackeval"}:
                if p.is_dir():
                    shutil.rmtree(p)
                else:
                    p.unlink()

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


In [ ]:
# [8] OLD A3 on validation — reference IDS

OLD_A3_VAL_ROOT = Path("/content/old_a3_validation")

old_a3_system = dict(A3_SYSTEM_TEMPLATE)
old_a3_system["name"] = "A3_OLD_SCI"

old_a3_val = run_single_system(
    dataset=VAL_DIR,
    manifest=VAL_MANIFEST,
    system=old_a3_system,
    run_root=OLD_A3_VAL_ROOT,
    controller_factory=OldPresentationController,
    keep_tracks=False,
)

print("\n=== OLD A3 VALIDATION ===")
print(json.dumps(old_a3_val, indent=2))

BASELINE_IDS = old_a3_val["IDS"]

baseline_path = RESULT_ROOT / "OLD_A3_VALIDATION.json"
baseline_path.write_text(json.dumps(old_a3_val, indent=2))

print("IDS ceiling:", BASELINE_IDS)
print("Saved:", baseline_path)


In [ ]:
# [9] Optuna multi-objective search on validation only

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState

TRIAL_WORK_ROOT = Path("/content/acmot_optuna_sci_trials")

if TRIAL_WORK_ROOT.exists():
    shutil.rmtree(TRIAL_WORK_ROOT)

TRIAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)

def objective(trial):
    weights = normalized_weights_from_trial(trial)

    system = dict(A3_SYSTEM_TEMPLATE)
    system["name"] = f"TRIAL_{trial.number:03d}"

    trial_root = TRIAL_WORK_ROOT / system["name"]
    controller_factory = lambda spec: TunableSCIController(spec, weights)

    try:
        result = run_single_system(
            dataset=VAL_DIR,
            manifest=VAL_MANIFEST,
            system=system,
            run_root=trial_root,
            controller_factory=controller_factory,
            keep_tracks=False,
        )

    except Exception as exc:
        trial.set_user_attr("error", repr(exc))
        print(f"[TRIAL {trial.number}] FAILED:", repr(exc))
        return -1.0, 10**9, 0.0

    trial.set_user_attr("weights", weights)

    for key, value in result.items():
        if isinstance(value, (int, float, str, bool)):
            trial.set_user_attr(key, value)

    print(
        f"[TRIAL {trial.number:03d}] "
        f"MOTA={100*result['MOTA']:.3f} "
        f"IDS={result['IDS']} "
        f"FPS={result['FPS']:.2f} "
        f"weights={weights}"
    )

    # 1) maximize MOTA
    # 2) minimize IDS
    # 3) maximize FPS
    return result["MOTA"], result["IDS"], result["FPS"]

sampler = TPESampler(seed=SEED, multivariate=True)

study = optuna.create_study(
    directions=["maximize", "minimize", "maximize"],
    sampler=sampler,
    study_name="acmot_sci_only_validation",
)

def export_callback(study, trial):
    df = study.trials_dataframe(
        attrs=("number", "values", "params", "user_attrs", "state")
    )
    df.to_csv(
        RESULT_ROOT / "OPTUNA_SCI_VALIDATION_TRIALS.csv",
        index=False
    )

study.optimize(
    objective,
    n_trials=N_TRIALS,
    gc_after_trial=True,
    show_progress_bar=True,
    callbacks=[export_callback],
)

print("[OK] Optuna validation search finished")


In [ ]:
# [10] Freeze the final SCI weights using validation only

complete = [
    t for t in study.trials
    if t.state == TrialState.COMPLETE
    and t.values is not None
    and float(t.values[0]) >= 0.0
]

if not complete:
    raise RuntimeError("No successful Optuna trials")

pareto = list(study.best_trials)

def realtime_ok(t):
    return float(t.values[2]) >= MIN_FPS

def ids_ok(t):
    if not ENFORCE_IDS_NOT_WORSE_THAN_OLD_A3:
        return True
    return int(t.values[1]) <= int(BASELINE_IDS)

# Prefer Pareto-optimal solutions satisfying both constraints.
feasible = [
    t for t in pareto
    if realtime_ok(t) and ids_ok(t)
]

# If none are on the Pareto front, search all successful trials.
if not feasible:
    feasible = [
        t for t in complete
        if realtime_ok(t) and ids_ok(t)
    ]

if not feasible:
    raise RuntimeError(
        "No trial achieved both FPS>=25 and IDS<=old A3. "
        "Increase N_TRIALS or change the SCI/controller design; "
        "do not silently weaken the constraints."
    )

# Final deterministic ranking:
# highest MOTA -> lowest IDS -> highest FPS
best_trial = max(
    feasible,
    key=lambda t: (
        float(t.values[0]),
        -int(t.values[1]),
        float(t.values[2]),
    )
)

BEST_WEIGHTS = normalized_weights_from_params(best_trial.params)

FROZEN = {
    "protocol": "SCI-only tuning with fixed YOLOv8n and fixed ByteTrack",
    "trial_number": int(best_trial.number),
    "selection_rule": (
        "Validation only: FPS>=25 and IDS<=old-A3, "
        "then highest MOTA, lowest IDS, highest FPS."
    ),
    "weights": BEST_WEIGHTS,
    "validation": {
        "MOTA": float(best_trial.values[0]),
        "IDS": int(best_trial.values[1]),
        "FPS": float(best_trial.values[2]),
        "old_A3_IDS": int(BASELINE_IDS),
    },
    "detector": "fixed yolov8n.pt",
    "tracker": "fixed tuned ByteTrack for A3",
    "what_changed": "SCI weights only",
}

FROZEN_PATH = RESULT_ROOT / "FROZEN_OPTUNA_SCI_ONLY.json"
FROZEN_PATH.write_text(json.dumps(FROZEN, indent=2))

print("\n=== FROZEN SCI WEIGHTS ===")
print(json.dumps(FROZEN, indent=2))
print("Saved:", FROZEN_PATH)


In [ ]:
# [11] FINAL TEST — frozen SCI only, no retraining, no retuning
#
# Run-All safe:
# - If a completed final test already exists, this cell SKIPS rerunning test-dev
#   and displays the saved result.
# - It will not silently use test-dev repeatedly for tuning.

from datetime import datetime, timezone

TEST_LOCK = RESULT_ROOT / "FINAL_TEST_DONE.json"

if TEST_LOCK.exists():
    lock = json.loads(TEST_LOCK.read_text())
    saved_folder = Path(lock["final_result_folder"])
    saved_csv = saved_folder / "FINAL_TEST_COMPARISON.csv"

    print("[SKIP] Final test was already completed.")
    print("Lock:", TEST_LOCK)
    print("Saved result folder:", saved_folder)

    if saved_csv.exists():
        final_df = pd.read_csv(saved_csv)
        print("\n=== EXISTING FINAL TEST COMPARISON ===")
        display(final_df)
    else:
        print("[WARNING] Lock exists but CSV was not found:", saved_csv)

else:
    frozen = json.loads(FROZEN_PATH.read_text())
    FROZEN_WEIGHTS = frozen["weights"]

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    FINAL_ROOT = RESULT_ROOT / f"FINAL_TEST_{stamp}"
    FINAL_ROOT.mkdir(parents=True, exist_ok=False)

    systems = []

    a0 = dict(A0_SYSTEM_TEMPLATE)
    a0["name"] = "A0_DEFAULT_FIXED640"
    systems.append((a0, OldPresentationController))

    a3_old = dict(A3_SYSTEM_TEMPLATE)
    a3_old["name"] = "A3_OLD_SCI"
    systems.append((a3_old, OldPresentationController))

    a3_new = dict(A3_SYSTEM_TEMPLATE)
    a3_new["name"] = "A3_OPTUNA_SCI_FROZEN"
    systems.append(
        (a3_new, lambda spec: TunableSCIController(spec, FROZEN_WEIGHTS))
    )

    final_rows = []

    for idx, (system, controller_factory) in enumerate(systems, 1):
        name = system["name"]
        print("\n" + "=" * 90)
        print(f"FINAL TEST {idx}/{len(systems)}: {name}")
        print("=" * 90)

        run_root = Path(f"/content/{name}_final_run")

        result = run_single_system(
            dataset=TEST_DIR,
            manifest=TEST_MANIFEST,
            system=system,
            run_root=run_root,
            controller_factory=controller_factory,
            keep_tracks=False,
        )

        final_rows.append({
            "System": name,
            "MOTA": 100 * result["MOTA"],
            "HOTA": 100 * result["HOTA"],
            "IDF1": 100 * result["IDF1"],
            "IDS": result["IDS"],
            "FP": result["FP"],
            "FN": result["FN"],
            "FPS": result["FPS"],
        })

    final_df = pd.DataFrame(final_rows)

    final_csv = FINAL_ROOT / "FINAL_TEST_COMPARISON.csv"
    final_json = FINAL_ROOT / "FINAL_TEST_COMPARISON.json"

    final_df.to_csv(final_csv, index=False)
    final_json.write_text(json.dumps(final_rows, indent=2))

    print("\n=== FINAL TEST COMPARISON ===")
    display(
        final_df.round({
            "MOTA": 3,
            "HOTA": 3,
            "IDF1": 3,
            "FPS": 3,
        })
    )

    TEST_LOCK.write_text(json.dumps({
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "final_result_folder": str(FINAL_ROOT),
        "frozen_weights_file": str(FROZEN_PATH),
        "detector": "fixed yolov8n.pt",
        "tracker": "unchanged",
        "changed_component": "SCI weights only",
    }, indent=2))

    print("\nSaved final table:", final_csv)
    print("Test lock:", TEST_LOCK)


# ما الذي تغيّر في هذه النسخة؟

فقط أوزان مؤشر تعقيد المشهد (SCI).

لم يتم:
- تدريب (YOLOv8n)
- تغيير أوزان الكاشف
- تغيير بنية الكاشف
- تغيير نوع المتعقب (ByteTrack)
- استخدام مجموعة الاختبار لاختيار المعاملات

وبالتالي إذا تحسنت النتائج، يكون تفسير التحسن أن طبقة (SCI) الجديدة واختيار أوزانها بالتحسين الآلي (Optuna) هي المتغير الأساسي في التجربة.

## ملاحظة عن VisDrone الرسمي

لأن الكاشف الثابت (YOLOv8n COCO) لا يحتوي فئة مستقلة لـ (van)، لا يجب تسمية هذه النتيجة:
**Full official 5-class VisDrone benchmark**

لكن استخدام (Validation / Test-dev) الرسميتين ما زال صحيحًا لتجربة رسالتك، مع الحفاظ على نفس بروتوكول التقييم المستخدم في مشروعك للمقارنة العادلة مع النتائج السابقة.
